In [1]:
from http import client

from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

True

In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient
client=MultiServerMCPClient(
    {
        "travel_server":{
            "transport":"streamable-http",
            "url":"https://mcp.kiwi.com",
        },
        "tavily_mcp":{
            "transport":"http",
            "url":"https://mcp.tavily.com/mcp/?tavilyApiKey=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"
        }
    }
)
tools=await client.get_tools()
print(tools)

[StructuredTool(name='search-flight', description='# Search for flights\n\nSearches Kiwi.com for available flights between two locations for the given dates and passengers. City or airport names are resolved automatically, so call this whenever the user wants to search for flights — whether they gave IATA codes or just place names.\n\n## Result shape\n\nReturns `{ query, currency, passengers, resultsCount, itineraries, searchTimeMs }`. Each item in `itineraries` has:\n- `price` (number) and `priceFormatted` (e.g. "123 EUR")\n- `totalDurationSeconds`\n- `bookingUrl` — the link to book the flight\n- `outbound` (and `inbound` for return flights), each a leg with: `route` (list of airport codes including layovers, e.g. ["PRG","MAD","BCN"]), `departureTime` / `arrivalTime` (local ISO timestamps), `durationSeconds`, `stops`, `cabinClass`, and a `segments` list.\n\n## How to display\n\nRender a markdown table. Group results into cheapest (lowest `price`), shortest (lowest `totalDurationSecond

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent=create_agent("groq:llama-3.1-8b-instant",
                   tools=tools,
                   checkpointer=InMemorySaver(),
                   system_prompt="you are a travel agent.no follow up questions")

In [8]:
from langchain.messages import HumanMessage
config={"configurable":{
    "thread_id":1
}}
response = await agent.ainvoke(
    {
        "messages": [HumanMessage(content="Can you get the weather report for hyd ")]
    },
    config=config
)